# Path train data generation

This notebook generates the paths that will be used for the path classification model training. It uses the segmentation predictions generated in the [previous notebook (3)](./03_generate_segmentation_preds.ipynb) to create the paths.

Different path generation methods can be used, but the one used in this notebook is the one detailed in the original paper, which consists in using the segmentation prediction and the ground truth to find the parts of vessels that have been forgotten by the segmentation model. The paths are then generated by connecting the endpoints of these parts to the rest of the vessel tree, and are constituted of a list of coordinates on the image, that follow the an euclidean shortest path between the endpoints.

In [1]:
from typing import List, Optional, Tuple
import numpy as np

In [2]:
import os
import json

data_dir_name = os.environ.get("EVAPORE_NOTEBOOKS_DATA_DIR", "FIVES")
data_dir_base = os.path.abspath('../data/')
data_dir = os.path.join(data_dir_base, data_dir_name)
print(data_dir)

split_filepath = os.path.join(data_dir, 'splits.json')

dataset_infos_filename = "dataset_infos.json"
dataset_infos_filepath = os.path.join(data_dir, dataset_infos_filename)
with open(dataset_infos_filepath, "r") as f:
    dataset_infos = json.load(f)
ndim = dataset_infos["ndim"]
print(ndim)

/home/morand/afs/EVAPORE/data/PERSEVERE_subset
3


In [3]:
gt_folder = os.path.join(data_dir, "gt")
pred_folder = os.path.join(data_dir, "pred")
img_folder = os.path.join(data_dir, "img")

gt_path_list = os.listdir(gt_folder)
gt_path_list.sort()

main_centerlines_folder = os.path.join(data_dir, "centerlines")
os.makedirs(main_centerlines_folder, exist_ok=True)

As stated in the paper, we only consider centerlines with an euclidean length less than 100 pixels, as longer centerlines are more likely to drift away from the euclidean path bewteen endpoints. Becasue of that, the feature sampling is more likely to be irrelevant and thus those paths are more likely to be wrongly labeled by the model.

If you want to use all the centerlines, just set max_dist to None.

For more informations about this choice, you can refer to the paper, or experiment it yourself by following the [centerline length study notebook (11)](11_centerline_length_study.ipynb).

In [4]:
max_dist = 100 # in pixels, you can change this value to generate centerlines with different distance thresholds

if max_dist is None:
    centerlines_folder = os.path.join(main_centerlines_folder, f"euclidean_all_centerlines")
else:
    centerlines_folder = os.path.join(main_centerlines_folder, f"euclidean_lt_{max_dist}_centerlines")
os.makedirs(centerlines_folder, exist_ok=True)

As stated in the paper, we use a euclidean distance-base path construction method to generate the centerlines.

You can replace it with other path construction methods, like the ones provided in the [path reconstruction methods folder](../src/utils/reconstruction/path_reconstruction/), or even implement one yourself, but as explained in the [reconstruction methods benchmark notebook (12)](./12_reconstruction_methods_benchmark.ipynb), the average distance from the ground truth path with the euclidean method is the lowest among all used methods, even more complex methods.

Some of the other methods may use more data to work, like probability maps, so you will need to adapt the code to load them as well.

In [5]:
from utils.reconstruction.path_reconstruction.euclidean_path_reconstruction import EuclideanPathReconstructionMethod

path_reconstruction_method = EuclideanPathReconstructionMethod()

# \#TODO

In [ ]:
from skimage.measure import label

def cut_mask_from_negative_edges(centerline: np.ndarray,    # (L, 2) or (L, 3)
                                  mask: np.ndarray          # (H, W) or (D, H, W)
) -> list:
    centerline_mask = np.zeros_like(mask, dtype=bool)
    for coords in centerline:
        centerline_mask[tuple(coords)] = True

    combined_mask = np.logical_and(centerline_mask, np.logical_not(mask))
    labeled_mask, num_labels = label(
        combined_mask, return_num=True, connectivity=mask.ndim
    )

    centerlines = [[] for _ in range(num_labels)]
    for coords in centerline:
        label_id = labeled_mask[tuple(coords)]
        if label_id > 0:
            centerlines[label_id - 1].append([int(c) for c in coords])
    return centerlines

def cut_mask_from_negative_edges_for_all(centerlines: list,  # list of (L, 2) or (L, 3)
                                         mask: np.ndarray,   # (H, W) or (D, H, W)
                                         classes: bool = None,
                                         edges: bool = None,
                                         return_old_centerlines: bool = False

) -> dict:
    
    new_centerlines = []
    new_classes = []
    new_edges = []
    old_centerlines = []
    for i, centerline in enumerate(centerlines):
        new_centerline = cut_mask_from_negative_edges(centerline, mask)
        n_new_centerline = len(new_centerline)
        if n_new_centerline > 0:
            new_centerlines.extend(new_centerline)
            if return_old_centerlines:
                old_centerlines.extend([centerline for _ in range(n_new_centerline)])
            if classes is not None:
                cls = classes[i]
                new_classes.extend([cls for _ in range(n_new_centerline)])
            if edges is not None:
                edge = edges[i]
                new_edges.extend([edge for _ in range(n_new_centerline)])

    res = {"new_centerlines": new_centerlines}
    if return_old_centerlines:
        res["old_centerlines"] = old_centerlines
    if classes is not None:
        res["classes"] = new_classes
    if edges is not None:
        res["edges"] = new_edges
    return res

In [7]:
from skimage.morphology import dilation, disk, ball

def clean_paths_on_surface_of_mask(centerlines,
                                   mask,
                                   kernel_size=1,
                                   threshold=0.5,
                                   classes=None
):
    
    new_centerlines = []
    if classes is not None:
        new_classes = []

    labeled_mask = label(mask)
    footprint = disk(kernel_size) if mask.ndim == 2 else ball(kernel_size)
    dilated_mask = dilation(labeled_mask, footprint)

    for i, centerline in enumerate(centerlines):
        centerline_mask = np.zeros_like(mask, dtype=np.uint8)
        for coords in centerline:
            centerline_mask[tuple(coords)] = 1

        combined_mask = centerline_mask * dilated_mask
        values_on_mask = np.count_nonzero(combined_mask)
        n_diff_cc = len(np.unique(combined_mask)) - (1 if 0 in combined_mask else 0)
        ratio_on_mask = values_on_mask / len(centerline)

        if ratio_on_mask <= threshold or n_diff_cc > 1:
            new_centerlines.append(centerline)
            if classes is not None:
                new_classes.append(classes[i])

    if classes is not None:
        return new_centerlines, new_classes
    return new_centerlines

In [8]:
def is_reconstructed_path_not_too_far(
        true_path_existing_centerline: np.ndarray,      # (M, 2) or (M, 3)
        true_path_reconstructed_centerline: np.ndarray,  # (N, 2) or (N, 3)
        distance_ratio_threshold: float
) -> bool:
    
    existing = np.asarray(true_path_existing_centerline, dtype=float)
    reconstructed = np.asarray(true_path_reconstructed_centerline, dtype=float)

    # pairwise distances: (N, M)
    diffs = reconstructed[:, None, :] - existing[None, :, :]
    dists = np.linalg.norm(diffs, axis=-1)
    min_dists = dists.min(axis=1)

    sum_min_distances = min_dists.sum() / len(reconstructed)
    return sum_min_distances <= distance_ratio_threshold


def remove_too_far_reconstructed_paths_for_all(
        true_path_existing_centerlines: List[np.ndarray],
        true_path_reconstructed_centerlines: List[np.ndarray],
        distance_ratio_threshold: float
) -> List[int]:
    
    new_reconstructed_classes = []
    for i, true_path_reconstructed_centerline in enumerate(true_path_reconstructed_centerlines):
        true_path_existing_centerline = true_path_existing_centerlines[i]
        condition = is_reconstructed_path_not_too_far(
            true_path_existing_centerline,
            true_path_reconstructed_centerline,
            distance_ratio_threshold
        )
        new_reconstructed_classes.append(int(condition))
    return new_reconstructed_classes

In [9]:
from typing import Dict, List, Optional
import networkx as nx
import numpy as np
import torch


def get_query_edges(
    graph: nx.Graph,
    n_closest: int = 1,
    max_dist: Optional[float] = None
) -> torch.Tensor:
    
    G = graph.copy()

    connected_components = list(nx.connected_components(graph))

    cc_extremities: Dict[int, List] = {}
    for cc_i, cc in enumerate(connected_components):
        extremities = []
        for n in cc:
            if G.degree(n) == 1:
                extremities.append(n)
        cc_extremities[cc_i] = extremities

    cc_n_count = [len(cc) for cc in connected_components]
    main_cc = np.argmax(cc_n_count)
    small_ccs = {i: cc for i, cc in enumerate(connected_components) if i != main_cc}

    virtual_edges_to_add: List[List] = []
    for i, cc in small_ccs.items():
        extremities = cc_extremities[i]
        other_ccs = {j: other_cc for j, other_cc in enumerate(connected_components) if j != i}
        other_ccs_all_nodes = []
        for other_cc in other_ccs.values():
            other_ccs_all_nodes.extend(other_cc)

        for extremity in extremities:
            other_nodes_distances = []
            extremity_pos = np.array(G.nodes[extremity]['pos'])
            for other_cc_node in other_ccs_all_nodes:
                other_cc_node_pos = np.array(G.nodes[other_cc_node]['pos'])
                dist = np.linalg.norm(extremity_pos - other_cc_node_pos)
                other_nodes_distances.append(dist)
            other_nodes_distances = np.array(other_nodes_distances)
            closest_indices = np.argsort(other_nodes_distances)[:n_closest]
            if max_dist is not None:
                closest_indices = closest_indices[other_nodes_distances[closest_indices] <= max_dist]
            closest_nodes = [other_ccs_all_nodes[idx] for idx in closest_indices]

            for extremity_closest_other_cc_node in closest_nodes:
                if (extremity, extremity_closest_other_cc_node) not in virtual_edges_to_add and \
                   (extremity_closest_other_cc_node, extremity) not in virtual_edges_to_add:
                    virtual_edges_to_add.append([extremity, extremity_closest_other_cc_node])

    virtual_edges_index_tensor = torch.tensor(virtual_edges_to_add, dtype=torch.long).t().contiguous()
    return virtual_edges_index_tensor

In [17]:
import json
import matplotlib.pyplot as plt
from PIL import Image

from graph.graph_pred_state import EdgePredState

def get_positive_samples(
    new_nx_graph: nx.Graph,
    pred_np: np.ndarray,
    max_dist: float
) -> Tuple[List[List[List[int]]], List[int]]:
    '''
    Computes positive samples for training the path reconstruction model.
    It identifies edges that are in the ground truth but not in the prediction and get the paths for these edges.
    The reconstructed paths are then filtered to get clean positive samples for training the model.

    Args:
        new_nx_graph (nx.Graph): The graph containing all the edges (both in prediction and not in prediction).
        pred_np (np.ndarray): The numpy array representing the prediction mask.
        max_dist (float): The maximum distance threshold to consider for true path edges.

    Returns:
        tuple[list, list]: A tuple containing the list of positive centerlines and their corresponding classes (all 1)
    '''
    true_path_edges = []
    for u, v, d in new_nx_graph.edges(data=True):
        edge_pred_state = d.get("edge_pred_state", None)
        u_pos, v_pos = new_nx_graph.nodes[u]['pos'], new_nx_graph.nodes[v]['pos']
        if edge_pred_state in [EdgePredState.NOT_IN_PREDICTION, EdgePredState.NOT_IN_PREDICTION.value]:
            euclidean_dist = np.linalg.norm(np.array(u_pos) - np.array(v_pos))
            if u_pos != v_pos and euclidean_dist <= max_dist:
                true_path_edges.append([u, v])

    true_path_edges = torch.tensor(true_path_edges, dtype=torch.long).t().contiguous()
    train_positive_centerlines = path_reconstruction_method.reconstruct(map=None, graph=new_nx_graph, new_edges=true_path_edges)
    train_positive_centerlines = [
        [[int(c) for c in coords] for coords in path]
        for path in train_positive_centerlines
    ]
    #train_positive_centerlines = cut_mask_from_negative_edges_for_all(train_positive_centerlines, pred_np)["new_centerlines"]
    #train_positive_centerlines = clean_paths_on_surface_of_mask(train_positive_centerlines, pred_np, kernel_size=2, threshold=0.5)
    train_positive_classes = [1] * len(train_positive_centerlines)

    return train_positive_centerlines, train_positive_classes


def get_negative_samples(
    new_nx_graph: nx.Graph,
    in_pred_graph: nx.Graph,
    pred_np: np.ndarray,
    gt_np: np.ndarray,
    max_dist: float,
    n_closest: int,
) -> Tuple[List[List[List[int]]], List[int]]:
    '''
    Computes negative samples for training the path reconstruction model.
    It identifies edges that are not present in the graph and get the paths for these edges.
    The reconstructed paths are then filtered to get clean negative samples for training the model.

    Args:
        new_nx_graph (nx.Graph): The graph containing all the edges (both in prediction and not in prediction).
        in_pred_graph (nx.Graph): The graph containing only the edges that are in the prediction.
        pred_np (np.ndarray): The numpy array representing the prediction mask.
        gt_np (np.ndarray): The numpy array representing the ground truth mask.
        max_dist (float): The maximum distance threshold to consider for negative path edges.
        n_closest (int): The number of closest nodes to consider when generating negative samples.

    Returns:
        tuple[list, list]: A tuple containing the list of negative centerlines and their corresponding classes (all 0)
    '''
    false_query_edges = get_query_edges(in_pred_graph, n_closest=n_closest, max_dist=max_dist).t().contiguous().tolist()
    train_negative_edges = []
    for (u, v) in false_query_edges:
        if not new_nx_graph.has_edge(u, v) and not new_nx_graph.has_edge(v, u):
            train_negative_edges.append([u, v])
    train_negative_edges = torch.tensor(train_negative_edges, dtype=torch.long).t().contiguous()

    train_negative_centerlines = path_reconstruction_method.reconstruct(map=None, graph=new_nx_graph, new_edges=train_negative_edges)
    train_negative_centerlines = [
        [[int(c) for c in coords] for coords in path]
        for path in train_negative_centerlines
    ]
    #train_negative_centerlines = cut_mask_from_negative_edges_for_all(train_negative_centerlines, pred_np)["new_centerlines"]
    #train_negative_centerlines = clean_paths_on_surface_of_mask(train_negative_centerlines, pred_np, kernel_size=2, threshold=0.5)
    #train_negative_centerlines = clean_paths_on_surface_of_mask(train_negative_centerlines, gt_np, kernel_size=0, threshold=0.5)
    train_negative_classes = [0] * len(train_negative_centerlines)

    return train_negative_centerlines, train_negative_classes

In [18]:
from image_segmentation.data.io_utils import load_array

from graph.graph_pred_state import get_combined_graph, get_combined_graph_optim
from graph.graph_oversampling import OversampleNodesTransform
from graph.graph_wrapper import GraphWrapper
from graph.graph_visualization import display_graph_overlay

def process_case(i: int, 
                 centerline_max_dist: float,
                 oversampling_max_dist: float,
                 n_closest: int,
                 display: bool = False,
                 use_optim_graph_creation: bool = True) -> None:
    '''
    Process a single case to generate training data for path reconstruction, and save the generated centerlines and their classes to a JSON file.

    Args:
        i (int): Index of the case to process.
        centerline_max_dist (float): The maximum distance threshold to consider for path when generating samples.
        oversampling_max_dist (float): The maximum distance threshold to consider when oversampling nodes in the graph.
        n_closest (int): The number of closest nodes to consider when generating negative samples.
        display (bool): Whether to display the case being processed (default: False).
    '''
    filename = gt_path_list[i]
    gt_path = os.path.join(gt_folder, filename)
    pred_path = os.path.join(pred_folder, filename)
    centerline_path = os.path.join(centerlines_folder, filename.replace(".png", ".json"))

    if os.path.exists(centerline_path):
        print(f"Centerline file already exists for {filename}, skipping...")
        return

    gt_np = load_array(gt_path, grayscale=True)
    pred_np = load_array(pred_path, grayscale=True)

    if gt_np.sum() == 0 or pred_np.sum() == 0:
        train_data = {"path_centerlines": [], "edges_classes": []}
        with open(centerline_path, 'w') as f:
            json.dump(train_data, f)
        print(f"No vessels in GT or prediction for {filename}, skipping...")
        return

    # Train data creation
    ccombined_graph: nx.Graph = None
    if use_optim_graph_creation:
        combined_graph = get_combined_graph_optim(gt_np, pred_np)
    else:
        combined_graph = get_combined_graph(gt_np, pred_np)
        
    oversample_nodes_transform = OversampleNodesTransform(oversampling_max_dist, remove_original_edges=True)
    graph_wrapper: GraphWrapper = oversample_nodes_transform(GraphWrapper(combined_graph))
    new_nx_graph: nx.Graph = graph_wrapper.get_graph()
    in_pred_graph: nx.Graph = graph_wrapper.in_pred_graph

    # Get positive and negative samples
    true_path_centerlines, true_edges_classes = get_positive_samples(new_nx_graph, pred_np, centerline_max_dist)
    print("positives samples done")
    train_negative_centerlines, train_negative_classes = get_negative_samples(new_nx_graph, in_pred_graph, pred_np, gt_np, centerline_max_dist, n_closest)
    print("negative samples done")

    if display:
        display_graph_overlay(gt_np, new_nx_graph, show_edges=True)
        

    # Combine positive and negative samples to create the training data
    train_data =  {
        "path_centerlines": train_negative_centerlines + true_path_centerlines,
        "edges_classes": train_negative_classes + true_edges_classes
    }

    # Save the training data to a JSON file
    with open(centerline_path, 'w') as f:
        json.dump(train_data, f, indent=4)


In [19]:
from tqdm import tqdm

oversampling_max_dist = 50.0
n_closest = 5

for i in tqdm(range(len(gt_path_list))):
    process_case(i, max_dist, oversampling_max_dist, n_closest, display=False)

  0%|          | 0/30 [00:00<?, ?it/s]

positives samples done
negative samples done


  7%|▋         | 2/30 [00:18<04:14,  9.10s/it]

positives samples done
negative samples done


 10%|█         | 3/30 [00:24<03:35,  7.97s/it]

...

 93%|█████████▎| 28/30 [04:02<00:16,  8.34s/it]

positives samples done
negative samples done
positives samples done
negative samples done


 97%|█████████▋| 29/30 [04:13<00:08,  8.97s/it]

positives samples done
negative samples done


100%|██████████| 30/30 [04:22<00:00,  8.75s/it]


In [20]:
print(f"Train data centerlines saved to {centerlines_folder}, total {len(os.listdir(centerlines_folder))} centerlines.")

Train data centerlines saved to /home/morand/afs/EVAPORE/data/PERSEVERE_subset/centerlines/euclidean_lt_100_centerlines, total 30 centerlines.


Now that we have the centerlines data to train the model on, you can continue on the [training path classification model notebook (5)](./05_train_path_classification_model.ipynb)